In [1]:
import os
import pandas as pd

selected_columns = [
    # Target
    "Severity",
    # Temporal
    "Start_Time",
    # Meteorological & Environment
    "Temperature(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "Precipitation(in)",
    "Weather_Condition",
    "Sunrise_Sunset",
    # Regional
    "State",
    # Road Infrastructure Flags
    "Amenity",
    "Bump",
    "Crossing",
    "Give_Way",
    "Junction",
    "No_Exit",
    "Railway",
    "Roundabout",
    "Stop",
    "Traffic_Signal",
]

input_file = "US_Accidents_March23.csv"
output_file = "US_Accidents_2016_2023_1M.csv"

# Historical start date
start_date = "2016-01-01"

# Clear destination file if it already exists
if os.path.exists(output_file):
    os.remove(output_file)

print("Starting streaming extraction for 1.0 Million records...")
total_saved_rows = 0
chunk_count = 0

# Stream through the source in 100,000-row chunks (RAM usage < 80 MB)
for chunk in pd.read_csv(
    input_file, usecols=selected_columns, chunksize=100_000, low_memory=False
):
    chunk_count += 1

    # Standardize datetime and remove rows missing the target
    chunk["Start_Time"] = pd.to_datetime(chunk["Start_Time"], errors="coerce")
    filtered = chunk[chunk["Start_Time"] >= start_date].dropna(
        subset=["Severity"]
    )

    if not filtered.empty:
        # Sample 13% per chunk across ~7.7M rows to hit ~1,000,000 records
        sampled = filtered.sample(frac=0.130, random_state=42)

        # Write header only on the very first written batch
        is_first_chunk = total_saved_rows == 0
        sampled.to_csv(
            output_file,
            mode="a",
            header=is_first_chunk,
            index=False,
        )

        total_saved_rows += len(sampled)

    print(
        f"Chunk {chunk_count:02d} processed | Rows saved: {total_saved_rows:,}",
        end="\r",
    )

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"\nExtraction completed successfully!")
print(f"Saved to: {output_file}")
print(f"Total Records Extracted: {total_saved_rows:,}")
print(f"File Size: {file_size_mb:.2f} MB")

Starting streaming extraction for 1.0 Million records...
Chunk 78 processed | Rows saved: 866,798
Extraction completed successfully!
Saved to: US_Accidents_2016_2023_1M.csv
Total Records Extracted: 866,798
File Size: 104.41 MB
